# Block 4: Model 2 Duration Band Classifier

This notebook trains the second supervised model in the EventOps AI pipeline.

**Goal:** Predict disruption duration band at event creation time.

Target classes:

- `short`: less than 1 hour
- `medium`: 1 to 4 hours
- `long`: more than 4 hours

Input:

- `outputs/model1_v2/model1_v2_duration_band_with_road_closure_features.csv`

This input includes the stacked Model 1 feature:

- `road_closure_probability`

Outputs:

- `outputs/model2_v2/model2_v2_duration_band_xgb_model.pkl`
- `outputs/model2_v2/model2_v2_test_predictions.csv`
- `outputs/model2_v2/model2_v2_feature_importance.csv`
- `outputs/model2_v2/model2_v2_metrics.json`
- `outputs/model2_v2/model2_v2_model_comparison.csv`

## 1. Imports and Paths

In [1]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except Exception as exc:
    CATBOOST_AVAILABLE = False
    CATBOOST_IMPORT_ERROR = exc

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 140)
pd.set_option('display.max_rows', 100)

RANDOM_STATE = 42

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'outputs' / 'model1_v2').exists()), cwd)
INPUT_PATH = PROJECT_ROOT / 'outputs' / 'model1_v2' / 'model1_v2_duration_band_with_road_closure_features.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'model2_v2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Input data:', INPUT_PATH)
print('Output dir:', OUTPUT_DIR)
print('CatBoost available:', CATBOOST_AVAILABLE)
if not CATBOOST_AVAILABLE:
    print('CatBoost import issue:', CATBOOST_IMPORT_ERROR)

Project root: /Users/astron_designer/GridLock_Phase2
Input data: /Users/astron_designer/GridLock_Phase2/outputs/model1_v2/model1_v2_duration_band_with_road_closure_features.csv
Output dir: /Users/astron_designer/GridLock_Phase2/outputs/model2_v2
CatBoost available: True


## 2. Load Stacked Duration Dataset

In [2]:
df = pd.read_csv(INPUT_PATH, low_memory=False)

print('Dataset shape:', df.shape)
print('Has road_closure_probability:', 'road_closure_probability' in df.columns)
print('Has duplicate predicted label:', 'road_closure_predicted_label' in df.columns)
print('Has duplicate percentage:', 'road_closure_percentage' in df.columns)
print('\nDuration band distribution:')
display(df['duration_band'].value_counts().to_frame('count'))
print('\nDuration band percentages:')
display((df['duration_band'].value_counts(normalize=True) * 100).round(2).to_frame('percent'))
display(df.head())

Dataset shape: (2764, 537)
Has road_closure_probability: True
Has duplicate predicted label: False
Has duplicate percentage: False

Duration band distribution:


,count
duration_band,
short,1581
medium,903
long,280



Duration band percentages:


,percent
duration_band,
short,57.20
medium,32.67
long,10.13


,latitude,longitude,valid_start_coordinate,has_start_location,distance_to_city_center_km,start_hour,start_dayofweek,start_weekofyear,is_weekend,is_peak_hour,is_night,hour_sin,hour_cos,day_sin,day_cos,report_lag_minutes_clipped,report_lag_hours_clipped,report_lag_is_negative,reporting_delay_minutes,text_length,description_char_length,description_word_count,has_non_ascii_text,has_kannada_text,has_accident_word,has_breakdown_word,has_water_word,has_construction_word,has_event_word,has_blocked_word,has_jam_word,has_vip_word,has_location_hint_word,has_emergency_word,is_planned_event,is_public_or_vip_event,is_breakdown_event,is_accident_event,is_weather_or_visibility_event,is_road_condition_event,has_vehicle_type,is_truck,is_bus,is_two_wheeler,is_heavy_vehicle,has_cargo_material,age_of_truck,truck_age_missing,past_count_event_cause,past_closure_rate_event_cause,past_count_corridor,past_closure_rate_corridor,past_count_zone,past_closure_rate_zone,past_count_junction,past_closure_rate_junction,past_count_police_station,past_closure_rate_police_station,past_count_location_grid,past_closure_rate_location_grid,past_count_event_cause_corridor,past_closure_rate_event_cause_corridor,location_grid_12_905_77_602,location_grid_12_919_77_589,location_grid_12_925_77_618,location_grid_12_927_77_621,location_grid_12_928_77_621,location_grid_12_931_77_687,location_grid_12_939_77_747,location_grid_12_942_77_747,...,corridor_peak_interaction_bellary_road_1_night,corridor_peak_interaction_bellary_road_1_off_peak,corridor_peak_interaction_bellary_road_2_morning_peak,corridor_peak_interaction_bellary_road_2_night,corridor_peak_interaction_bellary_road_2_off_peak,corridor_peak_interaction_cbd_2_night,corridor_peak_interaction_cbd_2_off_peak,corridor_peak_interaction_hennur_main_road_night,corridor_peak_interaction_hennur_main_road_off_peak,corridor_peak_interaction_hosur_road_morning_peak,corridor_peak_interaction_hosur_road_night,corridor_peak_interaction_hosur_road_off_peak,corridor_peak_interaction_irr_thanisandra_road_night,corridor_peak_interaction_irr_thanisandra_road_off_peak,corridor_peak_interaction_magadi_road_morning_peak,corridor_peak_interaction_magadi_road_night,corridor_peak_interaction_magadi_road_off_peak,corridor_peak_interaction_mysore_road_morning_peak,corridor_peak_interaction_mysore_road_night,corridor_peak_interaction_mysore_road_off_peak,corridor_peak_interaction_non_corridor_evening_peak,corridor_peak_interaction_non_corridor_morning_peak,corridor_peak_interaction_non_corridor_night,corridor_peak_interaction_non_corridor_off_peak,corridor_peak_interaction_non_corridor_unknown,corridor_peak_interaction_old_airport_road_night,corridor_peak_interaction_old_airport_road_off_peak,corridor_peak_interaction_old_madras_road_morning_peak,corridor_peak_interaction_old_madras_road_night,corridor_peak_interaction_old_madras_road_off_peak,corridor_peak_interaction_orr_east_1_morning_peak,corridor_peak_interaction_orr_east_1_night,corridor_peak_interaction_orr_east_1_off_peak,corridor_peak_interaction_orr_east_2_morning_peak,corridor_peak_interaction_orr_east_2_night,corridor_peak_interaction_orr_east_2_off_peak,corridor_peak_interaction_orr_north_1_morning_peak,corridor_peak_interaction_orr_north_1_night,corridor_peak_interaction_orr_north_1_off_peak,corridor_peak_interaction_orr_north_2_morning_peak,corridor_peak_interaction_orr_north_2_night,corridor_peak_interaction_orr_north_2_off_peak,corridor_peak_interaction_orr_west_1_morning_peak,corridor_peak_interaction_orr_west_1_night,corridor_peak_interaction_orr_west_1_off_peak,corridor_peak_interaction_other_rare,corridor_peak_interaction_tumkur_road_morning_peak,corridor_peak_interaction_tumkur_road_night,corridor_peak_interaction_tumkur_road_off_peak,corridor_peak_interaction_varthur_road_night,corridor_peak_interaction_varthur_road_off_peak,corridor_peak_interaction_west_of_chord_road_night,corridor_peak_interaction_west_of_chord_road_off_peak,start_day_name_friday,start_day_name_monday,sta

## 3. Feature Matrix and Target Encoding

We keep `road_closure_probability` as the stacked Model 1 signal and drop only the target.

In [3]:
target_col = 'duration_band'
assert target_col in df.columns, f'Missing target column: {target_col}'
assert 'road_closure_probability' in df.columns, 'Expected stacked feature road_closure_probability is missing'

X = df.drop(columns=[target_col]).copy()
y_raw = df[target_col].astype(str).str.lower().str.strip()

for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce')
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True)).fillna(0)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
inverse_label_mapping = {int(v): k for k, v in label_mapping.items()}
feature_cols = X.columns.tolist()

print('Feature matrix:', X.shape)
print('Label mapping:', label_mapping)
print('Missing values after imputation:', int(X.isna().sum().sum()))

Feature matrix: (2764, 536)
Label mapping: {'long': np.int64(0), 'medium': np.int64(1), 'short': np.int64(2)}
Missing values after imputation: 0


## 4. Train / Validation / Test Split

We use a stratified 70/15/15 split to preserve the rare `long` class in each split.

In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

print('Train:', X_train.shape, pd.Series(y_train).map(inverse_label_mapping).value_counts().to_dict())
print('Val:  ', X_val.shape, pd.Series(y_val).map(inverse_label_mapping).value_counts().to_dict())
print('Test: ', X_test.shape, pd.Series(y_test).map(inverse_label_mapping).value_counts().to_dict())

Train: (1934, 536) {'short': 1106, 'medium': 632, 'long': 196}
Val:   (415, 536) {'short': 238, 'medium': 135, 'long': 42}
Test:  (415, 536) {'short': 237, 'medium': 136, 'long': 42}


## 5. Evaluation Helper

In [5]:
def evaluate_model(name, model, X_eval, y_eval):
    pred = model.predict(X_eval)
    pred = np.asarray(pred).astype(int).ravel()
    proba = model.predict_proba(X_eval) if hasattr(model, 'predict_proba') else None

    metrics = {
        'model': name,
        'accuracy': float(accuracy_score(y_eval, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_eval, pred)),
        'macro_f1': float(f1_score(y_eval, pred, average='macro')),
        'weighted_f1': float(f1_score(y_eval, pred, average='weighted')),
    }
    precision, recall, f1, support = precision_recall_fscore_support(y_eval, pred, zero_division=0)
    per_class = pd.DataFrame({
        'class': label_encoder.classes_,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'support': support,
    })

    print(f'{name}')
    for k, v in metrics.items():
        if k != 'model':
            print(f'{k}: {v:.4f}')
    print('\nClassification Report:')
    print(classification_report(y_eval, pred, target_names=label_encoder.classes_))
    print('Confusion Matrix:')
    print(confusion_matrix(y_eval, pred))
    return metrics, per_class, pred, proba

## 6. Baseline: Balanced Random Forest

In [6]:
rf_model = RandomForestClassifier(
    n_estimators=500,
    min_samples_leaf=2,
    class_weight='balanced_subsample',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
rf_val_metrics, rf_val_per_class, rf_val_pred, rf_val_proba = evaluate_model(
    'RandomForest', rf_model, X_val, y_val
)
display(rf_val_per_class)

RandomForest
accuracy: 0.6241
balanced_accuracy: 0.6072
macro_f1: 0.5593
weighted_f1: 0.5937

Classification Report:
              precision    recall  f1-score   support

        long       0.51      0.76      0.61        42
      medium       0.49      0.24      0.33       135
       short       0.68      0.82      0.74       238

    accuracy                           0.62       415
   macro avg       0.56      0.61      0.56       415
weighted avg       0.60      0.62      0.59       415

Confusion Matrix:
[[ 32   3   7]
 [ 19  33  83]
 [ 12  32 194]]


,class,precision,recall,f1,support
0,long,0.507937,0.761905,0.609524,42
1,medium,0.485294,0.244444,0.325123,135
2,short,0.683099,0.815126,0.743295,238


## 7. Main Candidate: XGBoost Multi-Class

In [7]:
train_sample_weight = compute_sample_weight(class_weight='balanced', y=y_train)

xgb_model = XGBClassifier(
    objective='multi:softprob',
    num_class=len(label_encoder.classes_),
    eval_metric='mlogloss',
    n_estimators=800,
    max_depth=4,
    learning_rate=0.035,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2,
    reg_lambda=3.0,
    reg_alpha=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method='hist',
)

xgb_model.fit(
    X_train,
    y_train,
    sample_weight=train_sample_weight,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

xgb_val_metrics, xgb_val_per_class, xgb_val_pred, xgb_val_proba = evaluate_model(
    'XGBoost', xgb_model, X_val, y_val
)
display(xgb_val_per_class)

XGBoost
accuracy: 0.5831
balanced_accuracy: 0.5743
macro_f1: 0.5464
weighted_f1: 0.5798

Classification Report:
              precision    recall  f1-score   support

        long       0.48      0.67      0.56        42
      medium       0.41      0.36      0.39       135
       short       0.69      0.69      0.69       238

    accuracy                           0.58       415
   macro avg       0.53      0.57      0.55       415
weighted avg       0.58      0.58      0.58       415

Confusion Matrix:
[[ 28  10   4]
 [ 17  49  69]
 [ 13  60 165]]


,class,precision,recall,f1,support
0,long,0.482759,0.666667,0.560000,42
1,medium,0.411765,0.362963,0.385827,135
2,short,0.693277,0.693277,0.693277,238


## 8. Safety Candidate: CatBoost

CatBoost often gives strong minority/long-duration recall. We compare it if installed.

In [8]:
if CATBOOST_AVAILABLE:
    cat_model = CatBoostClassifier(
        loss_function='MultiClass',
        eval_metric='TotalF1',
        iterations=800,
        learning_rate=0.04,
        depth=5,
        l2_leaf_reg=5,
        random_seed=RANDOM_STATE,
        verbose=100,
        auto_class_weights='Balanced',
    )
    cat_model.fit(
        X_train,
        y_train,
        eval_set=(X_val, y_val),
        use_best_model=True,
    )
    cat_val_metrics, cat_val_per_class, cat_val_pred, cat_val_proba = evaluate_model(
        'CatBoost', cat_model, X_val, y_val
    )
    display(cat_val_per_class)
else:
    cat_model = None
    cat_val_metrics = None
    cat_val_per_class = None
    print('Skipping CatBoost because it is not installed.')

0:	learn: 0.6363272	test: 0.5936311	best: 0.5936311 (0)	total: 69.4ms	remaining: 55.5s


100:	learn: 0.6648294	test: 0.5865801	best: 0.6143614 (55)	total: 361ms	remaining: 2.5s


200:	learn: 0.7099722	test: 0.5878418	best: 0.6143614 (55)	total: 715ms	remaining: 2.13s


300:	learn: 0.7499162	test: 0.5769738	best: 0.6143614 (55)	total: 981ms	remaining: 1.63s


400:	learn: 0.7901343	test: 0.6029602	best: 0.6143614 (55)	total: 1.24s	remaining: 1.24s


500:	learn: 0.8138127	test: 0.5914003	best: 0.6143614 (55)	total: 1.52s	remaining: 906ms


600:	learn: 0.8353922	test: 0.5999615	best: 0.6143614 (55)	total: 1.78s	remaining: 589ms


700:	learn: 0.8581496	test: 0.5940353	best: 0.6143614 (55)	total: 2.03s	remaining: 287ms


799:	learn: 0.8732850	test: 0.5995565	best: 0.6143614 (55)	total: 2.33s	remaining: 0us

bestTest = 0.614361378
bestIteration = 55

Shrink model to first 56 iterations.
CatBoost
accuracy: 0.4988
balanced_accuracy: 0.6368
macro_f1: 0.5000
weighted_f1: 0.5000

Classification Report:
              precision    recall  f1-score   support

        long       0.34      0.95      0.51        42
      medium       0.41      0.59      0.49       135
       short       0.82      0.37      0.51       238

    accuracy                           0.50       415
   macro avg       0.53      0.64      0.50       415
weighted avg       0.64      0.50      0.50       415

Confusion Matrix:
[[ 40   0   2]
 [ 38  80  17]
 [ 38 113  87]]


,class,precision,recall,f1,support
0,long,0.344828,0.952381,0.506329,42
1,medium,0.414508,0.592593,0.487805,135
2,short,0.820755,0.365546,0.505814,238


## 9. Model Selection

Primary selection uses validation Macro-F1. We also keep balanced accuracy and long-class recall for operational interpretation.

In [9]:
comparison_rows = [rf_val_metrics, xgb_val_metrics]
if cat_val_metrics is not None:
    comparison_rows.append(cat_val_metrics)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values(['macro_f1', 'balanced_accuracy'], ascending=False).reset_index(drop=True)
display(comparison_df)

best_model_name = comparison_df.iloc[0]['model']
if best_model_name == 'XGBoost':
    final_model = xgb_model
elif best_model_name == 'CatBoost':
    final_model = cat_model
else:
    final_model = rf_model

print('Selected final model:', best_model_name)

,model,accuracy,balanced_accuracy,macro_f1,weighted_f1
0,RandomForest,0.624096,0.607158,0.559314,0.593725
1,XGBoost,0.583133,0.574302,0.546368,0.579775
2,CatBoost,0.498795,0.636840,0.499983,0.500008


Selected final model: RandomForest


## 10. Final Test Evaluation

In [10]:
test_metrics, test_per_class, test_pred, test_proba = evaluate_model(
    best_model_name, final_model, X_test, y_test
)

test_metrics.update({
    'selected_model': best_model_name,
    'feature_count': int(X.shape[1]),
    'train_rows': int(len(X_train)),
    'validation_rows': int(len(X_val)),
    'test_rows': int(len(X_test)),
    'used_road_closure_probability': bool('road_closure_probability' in X.columns),
    'label_mapping': {k: int(v) for k, v in label_mapping.items()},
})

display(test_per_class)

RandomForest
accuracy: 0.6096
balanced_accuracy: 0.5711
macro_f1: 0.5413
weighted_f1: 0.5778

Classification Report:
              precision    recall  f1-score   support

        long       0.53      0.67      0.59        42
      medium       0.43      0.23      0.30       136
       short       0.67      0.82      0.74       237

    accuracy                           0.61       415
   macro avg       0.54      0.57      0.54       415
weighted avg       0.58      0.61      0.58       415

Confusion Matrix:
[[ 28   8   6]
 [ 15  31  90]
 [ 10  33 194]]


,class,precision,recall,f1,support
0,long,0.528302,0.666667,0.589474,42
1,medium,0.430556,0.227941,0.298077,136
2,short,0.668966,0.818565,0.736243,237


## 11. Prediction Output

In [11]:
if test_proba is None:
    test_proba = np.zeros((len(X_test), len(label_encoder.classes_)))

proba_cols = [f'prob_{label}' for label in label_encoder.classes_]

test_predictions = pd.DataFrame(test_proba, columns=proba_cols, index=X_test.index)
test_predictions.insert(0, 'source_row_index', X_test.index)
test_predictions.insert(1, 'actual_duration_band', label_encoder.inverse_transform(y_test))
test_predictions.insert(2, 'predicted_duration_band', label_encoder.inverse_transform(np.asarray(test_pred).astype(int).ravel()))
test_predictions.insert(3, 'prediction_confidence', test_proba.max(axis=1))

test_predictions = test_predictions.sort_values('source_row_index').reset_index(drop=True)
display(test_predictions.head(20))

,source_row_index,actual_duration_band,predicted_duration_band,prediction_confidence,prob_long,prob_medium,prob_short
0,3,short,short,0.671708,0.000000,0.328292,0.671708
1,4,long,medium,0.504827,0.122715,0.504827,0.372458
2,13,short,short,0.759408,0.003051,0.237541,0.759408
3,15,medium,short,0.626186,0.003580,0.370234,0.626186
4,16,long,long,0.486486,0.486486,0.294455,0.219059
5,36,long,long,0.997733,0.997733,0.000477,0.001789
6,37,short,long,0.458642,0.458642,0.282107,0.259252
7,39,short,medium,0.573516,0.007578,0.573516,0.418906
8,40,medium,long,0.427473,0.427473,0.270327,0.302200
9,41,short,short,0.552931,0.232211,0.214858,0.552931


## 12. Feature Importance

In [12]:
if hasattr(final_model, 'feature_importances_'):
    importances = final_model.feature_importances_
else:
    importances = np.zeros(len(feature_cols))

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances,
}).sort_values('importance', ascending=False).reset_index(drop=True)

display(feature_importance.head(40))

,feature,importance
0,past_closure_rate_event_cause,0.048060
1,past_count_event_cause,0.032252
2,is_breakdown_event,0.030199
3,event_cause_grouped_vehicle_breakdown,0.028882
4,veh_type_grouped_unknown,0.028012
5,past_closure_rate_event_cause_corridor,0.023039
6,distance_to_city_center_km,0.022390
7,has_vehicle_type,0.020013
8,longitude,0.019552
9,road_closure_probability,0.019159


## 13. Save Artifacts

In [13]:
model_path = OUTPUT_DIR / 'model2_v2_duration_band_model.pkl'
predictions_path = OUTPUT_DIR / 'model2_v2_test_predictions.csv'
importance_path = OUTPUT_DIR / 'model2_v2_feature_importance.csv'
metrics_path = OUTPUT_DIR / 'model2_v2_metrics.json'
comparison_path = OUTPUT_DIR / 'model2_v2_model_comparison.csv'
per_class_path = OUTPUT_DIR / 'model2_v2_test_per_class_metrics.csv'

model_artifact = {
    'model': final_model,
    'model_name': best_model_name,
    'feature_cols': feature_cols,
    'label_encoder': label_encoder,
    'label_mapping': label_mapping,
    'metrics': test_metrics,
}

joblib.dump(model_artifact, model_path)
test_predictions.to_csv(predictions_path, index=False)
feature_importance.to_csv(importance_path, index=False)
comparison_df.to_csv(comparison_path, index=False)
test_per_class.to_csv(per_class_path, index=False)
with open(metrics_path, 'w') as f:
    json.dump(test_metrics, f, indent=2)

print('Saved model:', model_path)
print('Saved predictions:', predictions_path)
print('Saved feature importance:', importance_path)
print('Saved metrics:', metrics_path)
print('Saved comparison:', comparison_path)
print('Saved per-class metrics:', per_class_path)

Saved model: /Users/astron_designer/GridLock_Phase2/outputs/model2_v2/model2_v2_duration_band_model.pkl
Saved predictions: /Users/astron_designer/GridLock_Phase2/outputs/model2_v2/model2_v2_test_predictions.csv
Saved feature importance: /Users/astron_designer/GridLock_Phase2/outputs/model2_v2/model2_v2_feature_importance.csv
Saved metrics: /Users/astron_designer/GridLock_Phase2/outputs/model2_v2/model2_v2_metrics.json
Saved comparison: /Users/astron_designer/GridLock_Phase2/outputs/model2_v2/model2_v2_model_comparison.csv
Saved per-class metrics: /Users/astron_designer/GridLock_Phase2/outputs/model2_v2/model2_v2_test_per_class_metrics.csv


## 14. Final Summary

In [14]:
print('=' * 72)
print('BLOCK 4 COMPLETE: MODEL 2 DURATION BAND CLASSIFIER')
print('=' * 72)
print('Selected model:', best_model_name)
print('Features:', X.shape[1])
print('Used road_closure_probability:', 'road_closure_probability' in X.columns)
print('Test Accuracy:', round(test_metrics['accuracy'], 4))
print('Test Balanced Accuracy:', round(test_metrics['balanced_accuracy'], 4))
print('Test Macro-F1:', round(test_metrics['macro_f1'], 4))
print('Test Weighted-F1:', round(test_metrics['weighted_f1'], 4))
print('\nPer-class metrics:')
display(test_per_class)
print('=' * 72)

BLOCK 4 COMPLETE: MODEL 2 DURATION BAND CLASSIFIER
Selected model: RandomForest
Features: 536
Used road_closure_probability: True
Test Accuracy: 0.6096
Test Balanced Accuracy: 0.5711
Test Macro-F1: 0.5413
Test Weighted-F1: 0.5778

Per-class metrics:


,class,precision,recall,f1,support
0,long,0.528302,0.666667,0.589474,42
1,medium,0.430556,0.227941,0.298077,136
2,short,0.668966,0.818565,0.736243,237
